# 03 — Model Comparison
Side-by-side evaluation of RF, CNN, and LSTM using saved checkpoints from `train.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from src.ingestion.data_loader import UCIHARDataLoader
from src.pipeline.preprocessor import SignalPreprocessor
from src.features.feature_extractor import FeatureExtractor
from src.models.rf_baseline import RFClassifier
from src.models.cnn_classifier import CNNClassifier, _CNNBackbone, DEVICE
from src.models.lstm_classifier import LSTMClassifier, _LSTMBackbone
from src.utils.config import Config

cfg = Config()
loader = UCIHARDataLoader(cfg)
X_train, X_test, y_train, y_test = loader.load()

pp = SignalPreprocessor()
X_train_pp = pp.fit_transform(X_train)
X_test_pp  = pp.transform(X_test)

fe = FeatureExtractor()
X_train_feat = fe.extract(X_train_pp)
X_test_feat  = fe.extract(X_test_pp)

print('Data and features loaded.')

## Load saved model checkpoints

In [ ]:
# Random Forest
rf_model = joblib.load('../outputs/rf_model.pkl')
rf_preds = rf_model.predict(X_test_feat)

# CNN
cnn_backbone = _CNNBackbone().to(DEVICE)
cnn_backbone.load_state_dict(torch.load('../outputs/cnn_best.pt', map_location=DEVICE))
cnn_backbone.eval()
cnn_wrapper = CNNClassifier()
cnn_wrapper.model = cnn_backbone
cnn_preds = cnn_wrapper.predict(X_test_pp)

# LSTM
lstm_backbone = _LSTMBackbone().to(DEVICE)
lstm_backbone.load_state_dict(torch.load('../outputs/lstm_best.pt', map_location=DEVICE))
lstm_backbone.eval()
lstm_wrapper = LSTMClassifier()
lstm_wrapper.model = lstm_backbone
lstm_preds = lstm_wrapper.predict(X_test_pp)

print('Models loaded.')

## Results Table

In [ ]:
import pandas as pd

rows = []
for name, preds in [('RF', rf_preds), ('CNN', cnn_preds), ('LSTM', lstm_preds)]:
    rows.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'F1 (weighted)': round(f1_score(y_test, preds, average='weighted'), 4),
    })

pd.DataFrame(rows).set_index('Model')

## Confusion Matrices

In [ ]:
class_names = list(cfg.activity_labels.values())
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, preds) in zip(axes, [('RF', rf_preds), ('CNN', cnn_preds), ('LSTM', lstm_preds)]):
    cm = confusion_matrix(y_test, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, ax=ax, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, cbar=False, linewidths=0.4)
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{name}  (acc={acc:.4f})', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Normalised Confusion Matrices — RF vs CNN vs LSTM', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Per-class F1 scores

In [ ]:
from sklearn.metrics import f1_score

per_class = {}
for name, preds in [('RF', rf_preds), ('CNN', cnn_preds), ('LSTM', lstm_preds)]:
    per_class[name] = f1_score(y_test, preds, average=None)

df_pc = pd.DataFrame(per_class, index=class_names)

ax = df_pc.plot(kind='bar', figsize=(12, 5), colormap='tab10', edgecolor='white')
ax.set_title('Per-class F1 Score — RF vs CNN vs LSTM', fontweight='bold')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Model')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

df_pc.round(4)